In [1]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
import os
from sklearn.preprocessing import LabelEncoder
os.chdir("../../data/input_datasets")

In [ ]:
hancock_path = "../embeddings/HANCOCK/embeddings"
hancock_targets = pd.read_csv("hancock_targets.csv", index_col=0)
for col in ['recurrence', 'survival_status']:
    le = LabelEncoder()
    hancock_targets[col] = le.fit_transform(hancock_targets[col])
print(hancock_targets)


            recurrence  days_to_recurrence  survival_status  \
patient_id                                                    
1                    0                 NaN                1   
2                    1               357.0                1   
3                    0                 NaN                1   
4                    0                 NaN                1   
5                    1              1202.0                0   
...                ...                 ...              ...   
759                  1               690.0                0   
760                  1               132.0                0   
761                  1               606.0                0   
762                  0                 NaN                1   
763                  0                 NaN                1   

             survival_status_with_cause  days_to_last_information  rfs_event  \
patient_id                                                                     
1                   

In [11]:
NUM_DIMENSION = 64
result_dict = {"Method": [], "Target": [], "Type": [], "Score": []}
for i in range(10):
    # Load embeddings of all three datasets.
    hancock_file = os.path.join(hancock_path, f"HANCOCK_samples_{NUM_DIMENSION}_{i}.tsv")
    hancock_umap_file = os.path.join(hancock_path, f"HANCOCK_UMAP_NANAware_{NUM_DIMENSION}_{i}.csv")
    
    hancock_df = pd.read_csv(hancock_file, sep='\t', index_col=0)
    hancock_umap = pd.read_csv(hancock_umap_file, index_col=0)
    
    hancock_df.index = hancock_df.index.astype(str)
    hancock_umap.index = hancock_umap.index.astype(str)
    hancock_targets.index = hancock_targets.index.astype(str)

    assert set(hancock_df.index)==set(hancock_umap.index), "Indices are uneqal"
    assert set(hancock_targets.index)==set(hancock_umap.index), "Indices of target are unequal"

    hancock_df = hancock_df.join(hancock_targets)
    hancock_umap = hancock_umap.join(hancock_targets)

    # Init regression and CV models.
    clf = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=1000
    )

    cv = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Separability analysis for recurrence.
    hancock_rec_mask = hancock_df["recurrence"].notna()
    hancock_rec_df = hancock_df.loc[hancock_rec_mask]
    hancock_rec_X = hancock_rec_df.iloc[:, :NUM_DIMENSION].values
    hancock_rec_y = hancock_rec_df["recurrence"].values
    hancock_umap_rec_mask = hancock_umap["recurrence"].notna()
    hancock_umap_rec_df = hancock_umap.loc[hancock_umap_rec_mask]
    hancock_umap_rec_X = hancock_umap_rec_df.iloc[:, :NUM_DIMENSION].values
    hancock_umap_rec_y = hancock_umap_rec_df["recurrence"].values
    pos_class_ratio = hancock_rec_y.mean()
    
    pome_rec_auc_scores = cross_val_score(
        clf,
        hancock_rec_X,
        hancock_rec_y,
        cv=cv,
        scoring="average_precision"
    )    
    
    umap_rec_auc_scores = cross_val_score(
        clf,
        hancock_umap_rec_X,
        hancock_umap_rec_y,
        cv=cv,
        scoring="average_precision"
    )    
    
    assert len(pome_rec_auc_scores)==len(umap_rec_auc_scores)
    for i in range(len(pome_rec_auc_scores)):
        pome_auc = pome_rec_auc_scores[i]
        umap_auc = umap_rec_auc_scores[i]
        result_dict["Method"].append("POME")
        result_dict["Target"].append("Recurrence")
        result_dict["Score"].append(pome_auc)
        result_dict["Type"].append("AP")
        result_dict["Method"].append("UMAP")
        result_dict["Target"].append("Recurrence")
        result_dict["Score"].append(umap_auc)
        result_dict["Type"].append("AP")
    
    result_dict["Method"].append("Naive")
    result_dict["Target"].append("Recurrence")
    result_dict["Score"].append(pos_class_ratio)
    result_dict["Type"].append("AP")
        
    # Separability analysis for survival_status.
    hancock_surv_mask = hancock_df["survival_status"].notna()
    hancock_surv_df = hancock_df.loc[hancock_surv_mask]
    hancock_surv_X = hancock_surv_df.iloc[:, :NUM_DIMENSION].values
    hancock_surv_y = hancock_surv_df["survival_status"].values
    hancock_umap_surv_mask = hancock_umap["survival_status"].notna()
    hancock_umap_surv_df = hancock_umap.loc[hancock_umap_surv_mask]
    hancock_umap_surv_X = hancock_umap_surv_df.iloc[:, :NUM_DIMENSION].values
    hancock_umap_surv_y = hancock_umap_surv_df["survival_status"].values
    pos_class_ratio = hancock_surv_y.mean()
    
    pome_surv_auc_scores = cross_val_score(
        clf,
        hancock_surv_X,
        hancock_surv_y,
        cv=cv,
        scoring="average_precision"
    )    
    
    umap_surv_auc_scores = cross_val_score(
        clf,
        hancock_umap_surv_X,
        hancock_umap_surv_y,
        cv=cv,
        scoring="average_precision"
    )    
    
    assert len(pome_surv_auc_scores)==len(umap_surv_auc_scores)
    for i in range(len(pome_surv_auc_scores)):
        pome_auc = pome_surv_auc_scores[i]
        umap_auc = umap_surv_auc_scores[i]
        result_dict["Method"].append("POME")
        result_dict["Target"].append("Survival")
        result_dict["Score"].append(pome_auc)
        result_dict["Type"].append("AP")
        result_dict["Method"].append("UMAP")
        result_dict["Target"].append("Survival")
        result_dict["Score"].append(umap_auc)
        result_dict["Type"].append("AP")
    
    result_dict["Method"].append("Naive")
    result_dict["Target"].append("Survival")
    result_dict["Score"].append(pos_class_ratio)
    result_dict["Type"].append("AP")
        
    # Separability analysis for RFS event.
    hancock_rfs_mask = hancock_df["rfs_event"].notna()
    hancock_rfs_df = hancock_df.loc[hancock_rfs_mask]
    hancock_rfs_X = hancock_rfs_df.iloc[:, :NUM_DIMENSION].values
    hancock_rfs_y = hancock_rfs_df["rfs_event"].values
    hancock_umap_rfs_mask = hancock_umap["rfs_event"].notna()
    hancock_umap_rfs_df = hancock_umap.loc[hancock_umap_rfs_mask]
    hancock_umap_rfs_X = hancock_umap_rfs_df.iloc[:, :NUM_DIMENSION].values
    hancock_umap_rfs_y = hancock_umap_rfs_df["rfs_event"].values
    pos_class_ratio = hancock_rfs_y.mean()
    
    pome_rfs_auc_scores = cross_val_score(
        clf,
        hancock_rfs_X,
        hancock_rfs_y,
        cv=cv,
        scoring="average_precision"
    )    
    
    umap_rfs_auc_scores = cross_val_score(
        clf,
        hancock_umap_rfs_X,
        hancock_umap_rfs_y,
        cv=cv,
        scoring="average_precision"
    )    
    
    assert len(pome_rfs_auc_scores)==len(umap_rfs_auc_scores)
    for i in range(len(pome_rfs_auc_scores)):
        pome_auc = pome_rfs_auc_scores[i]
        umap_auc = umap_rfs_auc_scores[i]
        result_dict["Method"].append("POME")
        result_dict["Target"].append("RFS Event")
        result_dict["Score"].append(pome_auc)
        result_dict["Type"].append("AP")
        result_dict["Method"].append("UMAP")
        result_dict["Target"].append("RFS Event")
        result_dict["Score"].append(umap_auc)
        result_dict["Type"].append("AP")
    
    result_dict["Method"].append("Naive")
    result_dict["Target"].append("RFS Event")
    result_dict["Score"].append(pos_class_ratio)
    result_dict["Type"].append("AP")
    
        
    # Separability analysis for days to death.
    reg = LinearRegression()

    cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
    )
    
    hancock_days_mask = hancock_df["days_to_rfs_event"].notna()
    hancock_days_df = hancock_df.loc[hancock_days_mask]

    hancock_days_X = hancock_days_df.iloc[:, :NUM_DIMENSION].values
    hancock_days_y = hancock_days_df["days_to_rfs_event"].values

    hancock_umap_days_mask = hancock_umap["days_to_rfs_event"].notna()
    hancock_umap_days_df = hancock_umap.loc[hancock_umap_days_mask]

    hancock_days_umap_X = hancock_umap_days_df.iloc[:, :NUM_DIMENSION].values
    hancock_days_umap_y = hancock_umap_days_df["days_to_rfs_event"].values
    
    pome_days_r2_scores = cross_val_score(
    reg,
    hancock_days_X,
    hancock_days_y,
    cv=cv,
    scoring="r2"
    )

    umap_days_r2_scores = cross_val_score(
    reg,
    hancock_days_umap_X,
    hancock_days_umap_y,
    cv=cv,
    scoring="r2"
    )
    
    assert len(pome_days_r2_scores) == len(umap_days_r2_scores)
    for i in range(len(pome_days_r2_scores)):
        result_dict["Method"].append("POME")
        result_dict["Target"].append("Days to RFS")
        result_dict["Score"].append(pome_days_r2_scores[i])
        result_dict["Type"].append("R2")

        result_dict["Method"].append("UMAP")
        result_dict["Target"].append("Days to RFS")
        result_dict["Score"].append(umap_days_r2_scores[i])
        result_dict["Type"].append("R2")

In [12]:
result_df = pd.DataFrame(result_dict)
result_df
result_df.to_csv(f"HANCOCK_regression_results_{NUM_DIMENSION}.csv", index=False)